# NOVA TERRA LABS — Data pipeline (Colab version)
**NASA Space Apps Challenge 2026** · Data workstream

This notebook writes the same `backend/app` code that goes into the GitHub repository and runs it in Colab. It generates:
- `sites.json` (data contract for the front-end, metrics with 2 decimals and the source of each value)
- a **similarity index (0–100)** between each site and its planetary target (Jezero crater on Mars, Shackleton rim at the lunar south pole), with terrain measured on real Mars/Moon DEMs using the same method as on Earth
- a **world map** and a rotatable **3D globe** with all sites
- an **interactive 3D terrain viewer for every site**: real Copernicus DEM relief with the Sentinel-2 true-colour image draped on it

| Source | Used for |
|---|---|
| Copernicus DEM GLO-30 (ESA, 30 m) | elevation, slope, roughness, profile, 3D relief |
| NASA POWER (MERRA-2), 2015–2024 | temperature and daily range (adjusted to site elevation, 6.5 °C/km) and precipitation, same source for all sites; Open-Meteo only as fallback |
| USGS CTX DTM (Mars, Jezero) · NASA LOLA (Moon, Shackleton rim) | terrain of the planetary targets |
| NASA LRO Diviner Polar Resource Product | annual max / average temperature at the lunar target (context) |
| Sentinel-2 L2A true colour (Copernicus/ESA) | satellite image on the 3D relief (visual only) |

No API keys needed. **Runtime → Run all.** Everything is shown inside Colab: results table, sources, world map, 3D globe and a site explorer (3D terrain + profile). The results are saved to **NovaTerra Labs / Resultados** on Google Drive; the last cell optionally downloads a zip.

> To run it locally instead, use the repository: `cd backend && pip install -r requirements.txt && python -m app.pipeline` (see `backend/README.md`).

In [ ]:
!pip install -q rasterio pystac-client pyproj
!mkdir -p backend/app/assets/topojson backend/tests
# world map for the 3D globe (sane-topojson 4.0.0, MIT) so the downloaded viewers also work offline
!wget -q -P backend/app/assets/topojson https://cdn.jsdelivr.net/npm/sane-topojson@4.0.0/dist/world_50m.json https://cdn.jsdelivr.net/npm/sane-topojson@4.0.0/dist/world_110m.json

In [ ]:
%%writefile backend/app/__init__.py



In [ ]:
%%writefile backend/app/sites.py
"""Candidate analog sites.

lat/lon are approximate centre points of the analysis square and must be
validated by the team before the final submission.
"""

HALF_KM = 5  # half the side of the analysis square (5 km -> 10 x 10 km)
CLIMATE_START, CLIMATE_END = "2015-01-01", "2024-12-31"

SITES = [
    # --- Mars analogs -------------------------------------------------------
    {"id": "pampa-la-joya", "name": "La Pampa de la Joya, Arequipa", "country": "Peru",
     "lat": -16.60, "lon": -71.95, "analog_of": "Mars", "target": "jezero"},
    {"id": "andahua", "name": "Andahua Volcanic Field, Arequipa", "country": "Peru",
     "lat": -15.42, "lon": -72.33, "analog_of": "Mars", "target": "jezero"},
    {"id": "atacama", "name": "Atacama Desert (Yungay)", "country": "Chile",
     "lat": -24.10, "lon": -70.00, "analog_of": "Mars", "target": "jezero"},
    {"id": "haughton", "name": "Haughton Crater", "country": "Canada",
     "lat": 75.38, "lon": -89.68, "analog_of": "Mars", "target": "jezero"},
    {"id": "makgadikgadi", "name": "Makgadikgadi Pans", "country": "Botswana",
     "lat": -20.75, "lon": 25.30, "analog_of": "Mars", "target": "jezero"},
    {"id": "mcmurdo", "name": "McMurdo Dry Valleys (Wright Valley)", "country": "Antarctica",
     "lat": -77.52, "lon": 161.60, "analog_of": "Mars", "target": "jezero"},
    # Added 14 Nov 2026 — lesser-known analogs with published support:
    # Campo de Piedra Pómez: Favaro et al. (2020) Icarus 346, 113765 (Mars-analogue aeolian landforms, yardangs).
    #   Centre validated on the satellite map (14 Nov 2026): square centred on the white pumice/yardang field.
    {"id": "campo-piedra-pomez", "name": "Campo de Piedra Pómez, Catamarca", "country": "Argentina",
     "lat": -26.63, "lon": -67.485, "analog_of": "Mars", "target": "jezero"},
    # Salar de Pajonales: Ca-sulfate salar studied as an analogue of early-Mars climate change
    #   (Frontiers in Astronomy and Space Sciences, 2022, doi:10.3389/fspas.2021.797591).
    {"id": "salar-pajonales", "name": "Salar de Pajonales", "country": "Chile",
     "lat": -25.14444, "lon": -68.82000, "analog_of": "Mars", "target": "jezero"},
    # --- Moon analogs -------------------------------------------------------
    # Askja caldera (Smithsonian GVP 373060), where Apollo crews trained in 1965 and 1967
    {"id": "iceland-highlands", "name": "Askja caldera, Icelandic Highlands", "country": "Iceland",
     "lat": 65.033, "lon": -16.783, "analog_of": "Moon", "target": "lunar-south-pole"},
    # New lunar candidates (Atacama, Chile)
    {"id": "cerro-chao", "name": "Cerro Chao dacite dome, Atacama", "country": "Chile",
     "lat": -22.117, "lon": -68.150, "analog_of": "Moon", "target": "lunar-south-pole"},
    {"id": "monturaqui", "name": "Monturaqui impact crater, Atacama", "country": "Chile",
     "lat": -23.9275, "lon": -68.2617, "analog_of": "Moon", "target": "lunar-south-pole"},
    {"id": "meteor-crater", "name": "Meteor Crater (Barringer), Arizona", "country": "USA",
     "lat": 35.027, "lon": -111.022, "analog_of": "Moon", "target": "lunar-south-pole"},
    {"id": "craters-of-the-moon", "name": "Craters of the Moon, Idaho", "country": "USA",
     "lat": 43.42, "lon": -113.55, "analog_of": "Moon", "target": "lunar-south-pole"},
    {"id": "lanzarote", "name": "Timanfaya lava fields, Lanzarote", "country": "Spain",
     "lat": 29.01, "lon": -13.75, "analog_of": "Moon", "target": "lunar-south-pole"},
    {"id": "mauna-kea", "name": "Mauna Kea, Hawaii", "country": "USA",
     "lat": 19.82, "lon": -155.47, "analog_of": "Moon", "target": "lunar-south-pole"},
]


In [ ]:
%%writefile backend/app/terrain.py
"""Terrain metrics from the Copernicus DEM GLO-30 (public COGs on AWS, no key)."""
import math

import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.windows import from_bounds
from scipy.ndimage import uniform_filter

from .sites import HALF_KM

COP_URL = "https://copernicus-dem-30m.s3.amazonaws.com/{n}/{n}.tif"


def tile_name(lat_floor, lon_floor):
    """1x1 degree tiles are named after their south-west corner."""
    ns = "N" if lat_floor >= 0 else "S"
    ew = "E" if lon_floor >= 0 else "W"
    return f"Copernicus_DSM_COG_10_{ns}{abs(lat_floor):02d}_00_{ew}{abs(lon_floor):03d}_00_DEM"


def bbox(lat, lon, half_km=HALF_KM):
    """Square of 2*half_km around the point -> (west, south, east, north) in degrees."""
    dlat = half_km / 110.574
    dlon = half_km / (111.320 * math.cos(math.radians(lat)))
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat


def load_dem(lat, lon):
    """Reads only the needed window of the DEM (COG files: the full tile is not downloaded)."""
    w, s, e, n = bbox(lat, lon)
    names = {tile_name(math.floor(la), math.floor(lo)) for la in (s, n) for lo in (w, e)}
    srcs = []
    for nm in sorted(names):
        try:
            srcs.append(rasterio.open(COP_URL.format(n=nm)))
        except Exception as ex:  # tile may not exist (ocean)
            print(f"   tile not available: {nm} ({ex.__class__.__name__})")
    if not srcs:
        raise RuntimeError("No DEM coverage for this site")
    try:
        if len(srcs) == 1:
            src = srcs[0]
            win = from_bounds(w, s, e, n, src.transform)
            z = src.read(1, window=win, boundless=True, fill_value=np.nan).astype("float64")
            tr = src.window_transform(win)
        else:
            z, tr = merge(srcs, bounds=(w, s, e, n), nodata=np.nan)
            z = z[0].astype("float64")
    finally:
        for s_ in srcs:
            s_.close()
    z[z < -1000] = np.nan
    return z, tr


def pixel_size_m(transform, lat):
    """(dx, dy) in metres for a lat/lon grid."""
    dy = abs(transform.e) * 110574.0
    dx = abs(transform.a) * 111320.0 * math.cos(math.radians(lat))
    return dx, dy


def metrics_from_grid(z, dx, dy):
    """Elevation stats, mean slope (deg), roughness (m) and a W-E profile through the centre.

    z: elevation grid (m); dx, dy: pixel size in metres. Used for Earth, Mars and Moon alike.
    Roughness = std. dev. of elevation relative to its 5x5-pixel moving average (~150 m at 30 m),
    i.e. local irregularity excluding the regional slope.
    """
    zf = np.where(np.isnan(z), np.nanmean(z), z)
    gy, gx = np.gradient(zf, dy, dx)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    resid = zf - uniform_filter(zf, size=5)
    mid = zf.shape[0] // 2
    prof = zf[mid, :]
    idx = np.linspace(0, len(prof) - 1, 100).round().astype(int)
    return {
        "elevation_mean_m": float(np.nanmean(z)),
        "elevation_min_m": float(np.nanmin(z)),
        "elevation_max_m": float(np.nanmax(z)),
        "slope_mean_deg": float(np.mean(slope)),
        "roughness_m": float(np.std(resid[2:-2, 2:-2])),
    }, {
        "distance_km": [round(float(i * dx / 1000), 3) for i in idx],
        "elevation_m": [round(float(prof[i]), 1) for i in idx],
    }


def terrain_metrics(z, transform, lat):
    """Same metrics for a lat/lon grid (Copernicus DEM)."""
    dx, dy = pixel_size_m(transform, lat)
    return metrics_from_grid(z, dx, dy)


In [ ]:
%%writefile backend/app/climate.py
"""Earth climate metrics, 2015–2024.

Primary source: NASA POWER Daily API, no key needed.
- Temperature: MERRA-2 T2M_MAX / T2M_MIN (≈50 km grid), adjusted from the grid-cell elevation to the
  site's mean DEM elevation with a standard lapse rate of 6.5 °C/km (the daily range is unchanged).
- Precipitation: MERRA-2 corrected precipitation (PRECTOTCORR), same source for all sites.
  (GPM IMERG was tested and discarded: it strongly overestimated rainfall at arid, coastal and island sites.)
Fallback: Open-Meteo Historical API (ERA5 / ERA5-Land), used only if NASA POWER fails.
The source of each value is recorded in sites.json.
"""
import time

import pandas as pd
import requests

from .sites import CLIMATE_START, CLIMATE_END

POWER_API = "https://power.larc.nasa.gov/api/temporal/daily/point"
OPEN_METEO_API = "https://archive-api.open-meteo.com/v1/archive"


def _summarise(tmax, tmin, precip_daily, dates):
    d = pd.DataFrame({"tmax": tmax, "tmin": tmin, "p": precip_daily}, index=pd.to_datetime(dates))
    if d[["tmax", "tmin"]].isna().all().any():
        raise RuntimeError("no temperature values for this point")
    annual = d["p"].groupby(d.index.year).sum()
    return {
        "temp_max_mean_c": float(d.tmax.mean()),
        "temp_min_mean_c": float(d.tmin.mean()),
        "temp_range_c": float((d.tmax - d.tmin).mean()),
        "temp_abs_min_c": float(d.tmin.min()),
        "temp_abs_max_c": float(d.tmax.max()),
        "precip_annual_mm": float(annual.mean()),
    }


def _get(url, params, retries=5):
    for attempt in range(retries):
        r = requests.get(url, params=params, timeout=180)
        if r.status_code == 429 or r.status_code >= 500:
            wait = 15 * (attempt + 1)
            print(f"   HTTP {r.status_code} from {url.split('/')[2]} -> waiting {wait}s ({attempt + 1}/{retries})")
            time.sleep(wait)
            continue
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
        return r.json()
    raise RuntimeError(f"{url.split('/')[2]} still unavailable after {retries} attempts")


LAPSE_RATE_C_PER_KM = 6.5


def _power(lat, lon, parameters, community, utc=False):
    params = {"parameters": parameters, "community": community, "latitude": lat, "longitude": lon,
              "start": CLIMATE_START.replace("-", ""), "end": CLIMATE_END.replace("-", ""), "format": "JSON"}
    if utc:
        params["time-standard"] = "UTC"
    js = _get(POWER_API, params)
    par = js["properties"]["parameter"]
    fill = js.get("header", {}).get("fill_value", -999)
    clean = {k: {d: (None if v in (fill, -999, -99) else v) for d, v in par[k].items()} for k in par}
    grid_elev = (js.get("geometry", {}).get("coordinates") or [None, None, None])[2]
    return clean, grid_elev


def nasa_power_metrics(lat, lon, site_elev_m=None):
    """Returns (metrics, sources_by_metric, info)."""
    t, grid_elev = _power(lat, lon, "T2M_MAX,T2M_MIN,PRECTOTCORR", "AG")
    dates = sorted(t["T2M_MAX"])
    adj = 0.0
    if site_elev_m is not None and grid_elev is not None:
        adj = -LAPSE_RATE_C_PER_KM * (site_elev_m - grid_elev) / 1000.0
    tmax = [None if t["T2M_MAX"][d] is None else t["T2M_MAX"][d] + adj for d in dates]
    tmin = [None if t["T2M_MIN"][d] is None else t["T2M_MIN"][d] + adj for d in dates]
    precip = [t["PRECTOTCORR"][d] for d in dates]
    m = _summarise(tmax, tmin, precip, dates)
    src = {k: "nasa-power" for k in m}
    info = {"climate_grid_elevation_m": grid_elev, "temperature_lapse_adjustment_c": round(adj, 2)}
    return m, src, info


def open_meteo_metrics(lat, lon):
    js = _get(OPEN_METEO_API, {
        "latitude": lat, "longitude": lon, "start_date": CLIMATE_START, "end_date": CLIMATE_END,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum", "timezone": "UTC"})
    d = js["daily"]
    return _summarise(d["temperature_2m_max"], d["temperature_2m_min"], d["precipitation_sum"], d["time"])


def climate_metrics(lat, lon, site_elev_m=None):
    """Returns (metrics, sources_by_metric, info). NASA POWER first, Open-Meteo only as a fallback."""
    try:
        return nasa_power_metrics(lat, lon, site_elev_m)
    except Exception as ex:
        print(f"   NASA POWER failed ({ex}); using Open-Meteo fallback")
        m = open_meteo_metrics(lat, lon)
        return m, {k: "open-meteo-era5" for k in m}, {"climate_fallback": str(ex)[:200]}


In [ ]:
%%writefile backend/app/imagery.py
"""True-colour satellite imagery from Sentinel-2 L2A (Copernicus, public COGs on AWS, no key).

Scenes are found through the Element 84 Earth Search STAC API. For each site the
least cloudy / least snowy recent scene that fully covers the analysis square is
selected, and its 10 m true-colour ("visual") image is resampled onto the DEM grid
so it can be draped over the 3D terrain.
"""
from datetime import date, timedelta

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds as transform_from_bounds

STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-l2a"


def _score(item):
    p = item.properties
    return (p.get("eo:cloud_cover", 100)
            + 0.5 * p.get("s2:snow_ice_percentage", 0)
            + 0.2 * p.get("s2:nodata_pixel_percentage", 0))


def find_scenes(bounds, years=3, max_cloud=20, max_items=100):
    """Candidate scenes ordered best first (those fully covering the square come first)."""
    from pystac_client import Client
    from shapely.geometry import box, shape

    aoi = box(*bounds)
    end = date.today()
    start = end - timedelta(days=365 * years)
    search = Client.open(STAC_URL).search(
        collections=[COLLECTION], intersects=aoi.__geo_interface__,
        datetime=f"{start.isoformat()}/{end.isoformat()}",
        query={"eo:cloud_cover": {"lt": max_cloud}}, max_items=max_items)
    items = list(search.items())
    full = sorted([i for i in items if shape(i.geometry).contains(aoi)], key=_score)
    partial = sorted([i for i in items if not shape(i.geometry).contains(aoi)], key=_score)
    return full + partial


def read_true_color(href, bounds, shape_hw):
    """Reads the image window covering `bounds` (lon/lat) resampled to shape (H, W)."""
    h, w_px = shape_hw
    dst_transform = transform_from_bounds(*bounds, w_px, h)
    with rasterio.open(href) as src, WarpedVRT(
            src, crs="EPSG:4326", transform=dst_transform, width=w_px, height=h,
            resampling=Resampling.average, src_nodata=0, nodata=0) as vrt:
        rgb = vrt.read([1, 2, 3])
    return rgb.astype("uint8")


def fetch_true_color(bounds, shape_hw, max_tries=5, max_nodata=0.05):
    """Returns (rgb uint8 array (3,H,W), metadata dict). Raises if no usable scene."""
    last_err = "no Sentinel-2 scenes found"
    for item in find_scenes(bounds)[:max_tries]:
        try:
            rgb = read_true_color(item.assets["visual"].href, bounds, shape_hw)
        except Exception as ex:
            last_err = f"{item.id}: {ex}"
            continue
        nodata = float(np.mean(rgb.sum(axis=0) == 0))
        if nodata > max_nodata:
            last_err = f"{item.id}: {nodata:.0%} of the square without data"
            continue
        p = item.properties
        return rgb, {
            "source": "sentinel-2-l2a",
            "scene_id": item.id,
            "date": str(p.get("datetime", ""))[:10],
            "cloud_cover_pct": round(float(p.get("eo:cloud_cover", 0)), 2),
        }
    raise RuntimeError(f"Sentinel-2 imagery unavailable ({last_err})")


In [ ]:
%%writefile backend/app/viewer.py
"""Interactive 3D terrain viewers (offline HTML, no server or API key needed).

Each site is rendered as a Plotly Mesh3d built from the real Copernicus DEM, with the
Sentinel-2 true-colour image draped on it. The page lets the user switch between
satellite and elevation colouring and change the vertical exaggeration.
"""
import html
import math
import os
import shutil

import numpy as np
import plotly.graph_objects as go

from .terrain import pixel_size_m

BG = "#050B18"        # deep navy / near-black
BTN = "#0891B2"       # button text: readable on both dark and highlighted buttons
PANEL = "#0B1428"
LIME = "#B6FF3B"      # primary accent
CYAN = "#22D3EE"      # secondary accent
TEXT = "#E6EDF5"
MUTED = "#8A97AB"

EXAGGERATIONS = (1, 2, 5, 10)


def _grid(z, transform, lat, max_px):
    step = max(1, int(math.ceil(max(z.shape) / max_px)))
    zs = z[::step, ::step]
    zs = np.where(np.isnan(zs), np.nanmean(zs), zs)
    dx, dy = pixel_size_m(transform, lat)
    h, w = zs.shape
    x = np.arange(w) * dx * step / 1000
    y = (h - 1 - np.arange(h)) * dy * step / 1000          # north at the top
    X, Y = np.meshgrid(x, y)
    idx = np.arange(h * w).reshape(h, w)
    a, b = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel()
    c, d = idx[1:, :-1].ravel(), idx[1:, 1:].ravel()
    faces = (np.concatenate([a, b]), np.concatenate([b, d]), np.concatenate([c, c]))
    return step, zs, X.ravel(), Y.ravel(), faces


def build_figure(site, z, transform, rgb=None, imagery_meta=None, max_px=220, exaggeration=2):
    step, zs, X, Y, (i, j, k) = _grid(z, transform, site["lat"], max_px)
    res_m = round(max(pixel_size_m(transform, site["lat"])) * step)
    Z = zs.ravel()
    width_km = max(X.max(), Y.max())
    relief_km = (Z.max() - Z.min()) / 1000
    hover = "x %{x:.2f} km<br>y %{y:.2f} km<br>elevation %{z:.2f} m<extra></extra>"
    light = dict(ambient=0.55, diffuse=0.85, roughness=0.9, specular=0.05, fresnel=0.1)

    traces = []
    has_img = rgb is not None
    if has_img:
        r, g, b = (rgb[ch, ::step, ::step][: zs.shape[0], : zs.shape[1]].ravel() for ch in range(3))
        colors = [f"#{rr:02x}{gg:02x}{bb:02x}" for rr, gg, bb in zip(r, g, b)]
        traces.append(go.Mesh3d(x=X, y=Y, z=Z, i=i, j=j, k=k, vertexcolor=colors,
                                lighting=light, lightposition=dict(x=-5e4, y=5e4, z=1e5),
                                hovertemplate=hover, name="Satellite"))
    traces.append(go.Mesh3d(x=X, y=Y, z=Z, i=i, j=j, k=k, intensity=Z, colorscale="Earth",
                            colorbar=dict(title=dict(text="m", font=dict(color=TEXT)),
                                          tickfont=dict(color=TEXT), len=0.6),
                            lighting=light, lightposition=dict(x=-5e4, y=5e4, z=1e5),
                            hovertemplate=hover, name="Elevation", visible=not has_img))
    fig = go.Figure(traces)

    def zratio(ex):
        return max(0.03, relief_km * ex / width_km)

    menus = [dict(
        type="buttons", direction="right", x=0.0, y=1.0, xanchor="left", yanchor="bottom",
        bgcolor=PANEL, bordercolor=LIME, font=dict(color=BTN, size=12), showactive=True,
        buttons=[dict(label=f"×{ex}", method="relayout",
                      args=[{"scene.aspectratio.z": zratio(ex)}]) for ex in EXAGGERATIONS],
        active=EXAGGERATIONS.index(exaggeration))]
    if has_img:
        menus.append(dict(
            type="buttons", direction="right", x=1.0, y=1.0, xanchor="right", yanchor="bottom",
            bgcolor=PANEL, bordercolor=CYAN, font=dict(color=BTN, size=12), showactive=True,
            buttons=[dict(label="Satellite", method="restyle", args=[{"visible": [True, False]}]),
                     dict(label="Elevation", method="restyle", args=[{"visible": [False, True]}])]))

    if has_img:
        img_txt = (f"Imagery: Sentinel-2 L2A true colour, {imagery_meta.get('date', '')} "
                   f"(scene {imagery_meta.get('scene_id', '')}), Copernicus/ESA")
    else:
        img_txt = "Imagery: not available for this site (elevation colouring only)"
    src_txt = (f"Elevation: Copernicus DEM GLO-30 (ESA), shown at ≈{res_m} m grid · {img_txt}"
               " · Vertical exaggeration is visual only; hover shows true elevation.")

    axis = dict(color=MUTED, gridcolor="#1C2A44", backgroundcolor=BG, showbackground=True,
                zerolinecolor="#1C2A44")
    fig.update_layout(
        title=dict(text=f"<b>{html.escape(site['name'])}</b>  ·  {site['country']}  ·  "
                        f"analog of {site['analog_of']}  ·  {site['lat']:.3f}, {site['lon']:.3f}",
                   font=dict(color=TEXT, size=16), x=0.01, y=0.97),
        paper_bgcolor=BG, plot_bgcolor=BG, font=dict(color=TEXT),
        margin=dict(l=0, r=0, t=90, b=30), updatemenus=menus,
        annotations=[dict(text=src_txt, x=0, y=-0.02, xref="paper", yref="paper",
                          showarrow=False, font=dict(color=MUTED, size=10), xanchor="left")],
        scene=dict(xaxis=dict(title="W → E (km)", **axis), yaxis=dict(title="S → N (km)", **axis),
                   zaxis=dict(title="Elevation (m)", nticks=4, **axis), bgcolor=BG,
                   aspectmode="manual",
                   aspectratio=dict(x=X.max() / width_km, y=Y.max() / width_km, z=zratio(exaggeration)),
                   camera=dict(eye=dict(x=0.95, y=-0.95, z=0.6))))
    return fig


def write_site_page(fig, path):
    # 'directory' -> plotly.min.js is copied once next to the pages: works offline
    fig.write_html(path, include_plotlyjs="directory", full_html=True,
                   default_width="100%", default_height="100vh",
                   config={"displaylogo": False, "responsive": True})
    with open(path, encoding="utf-8") as fh:   # full-window dark page
        page = fh.read()
    page = page.replace("<head>", f"<head><title>NOVA TERRA LABS · {fig.layout.title.text.split('</b>')[0].replace('<b>', '')}</title>"
                        f"<style>html,body{{margin:0;height:100%;background:{BG};overflow:hidden}}</style>", 1)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(page)


def write_index(sites, path):
    rows = []
    for s in sites:
        m = {k: v["value"] for k, v in s["metrics"].items()}
        f = lambda k: "—" if k not in m else f"{m[k]:,.2f}"
        link = (f'<a href="{s["viewer_3d"].split("/")[-1]}">3D ↗</a>' if s.get("viewer_3d")
                else '<span class="muted">n/a</span>')
        warn = ' <span class="warn" title="{}">⚠</span>'.format(html.escape("; ".join(s["errors"]))) if s["errors"] else ""
        rows.append(
            f"<tr><td>{html.escape(s['name'])}{warn}<div class='muted'>{s['country']}</div></td>"
            f"<td><span class='tag {s['analog_of'].lower()}'>{s['analog_of']}</span></td>"
            f"<td>{f('elevation_mean_m')}</td><td>{f('slope_mean_deg')}</td><td>{f('roughness_m')}</td>"
            f"<td>{f('temp_range_c')}</td><td>{f('precip_annual_mm')}</td>"
            f"<td>{'—' if (s.get('similarity') or {}).get('score') is None else format(s['similarity']['score'], '.2f')}</td><td>{link}</td></tr>")
    page = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<title>NOVA TERRA LABS · Site viewers</title>
<style>
body{{margin:0;background:{BG};color:{TEXT};font-family:system-ui,Segoe UI,Roboto,sans-serif;
background-image:linear-gradient(#0E1A33 1px,transparent 1px),linear-gradient(90deg,#0E1A33 1px,transparent 1px);background-size:40px 40px}}
header{{padding:28px 32px 8px}} h1{{margin:0;letter-spacing:.18em;font-size:22px}} h1 span{{color:{LIME}}}
.sub{{color:{MUTED};margin-top:6px;font-size:13px}}
table{{margin:18px 32px 32px;border-collapse:collapse;background:rgba(11,20,40,.85);border:1px solid #1C2A44;width:calc(100% - 64px)}}
th,td{{padding:10px 14px;border-bottom:1px solid #1C2A44;text-align:left;font-size:14px}}
th{{color:{MUTED};font-weight:500;font-size:12px;letter-spacing:.08em;text-transform:uppercase}}
td:nth-child(n+3):not(:last-child){{font-variant-numeric:tabular-nums}}
a{{color:{LIME};text-decoration:none;font-weight:600}} a:hover{{text-decoration:underline}}
.muted{{color:{MUTED};font-size:12px}} .warn{{color:#FFB020;cursor:help}}
.tag{{font-size:11px;padding:2px 8px;border:1px solid;letter-spacing:.06em}}
.tag.mars{{color:#FF7A45;border-color:#FF7A45}} .tag.moon{{color:{CYAN};border-color:{CYAN}}}
footer{{color:{MUTED};font-size:11px;padding:0 32px 24px}}
</style></head><body>
<header><h1>NOVA <span>TERRA</span> LABS</h1>
<div class="sub">Understanding Earth to prepare for exploration beyond it · 3D site viewers (10 × 10 km) · <a href="globe.html">3D globe ↗</a></div></header>
<table><thead><tr><th>Site</th><th>Analog</th><th>Mean elev. (m)</th><th>Slope (°)</th>
<th>Roughness (m)</th><th>Temp. range (°C)</th><th>Precip. (mm/yr)</th><th>Similarity</th><th>View</th></tr></thead>
<tbody>{''.join(rows)}</tbody></table>
<footer>Sources: Copernicus DEM GLO-30 (ESA) · Sentinel-2 L2A (Copernicus/ESA) · Open-Meteo / ERA5 (ECMWF).
Full references in sites.json.</footer></body></html>"""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(page)


# ---------------------------------------------------------------------------
# 3D globe with all candidate sites
# ---------------------------------------------------------------------------
MARS = "#FF7A45"
LABEL_POS = {"pampa-la-joya": "middle right", "andahua": "middle left"}   # nearby sites


def _hover(s):
    m = {k: v["value"] for k, v in s.get("metrics", {}).items()}
    f = lambda k, u: f"{m[k]:,.2f} {u}" if k in m else "—"
    return (f"<b>{html.escape(s['name'])}</b><br>{s['country']} · analog of {s['analog_of']}"
            f"<br>{s['lat']:.3f}, {s['lon']:.3f}"
            f"<br>Mean elevation: {f('elevation_mean_m', 'm')}"
            f"<br>Mean slope: {f('slope_mean_deg', '°')}"
            f"<br>Roughness: {f('roughness_m', 'm')}"
            f"<br>Temp. range: {f('temp_range_c', '°C')}"
            f"<br>Precipitation: {f('precip_annual_mm', 'mm/yr')}"
            + (f"<br><b>Similarity: {s['similarity']['score']:.2f} / 100</b>"
               if (s.get('similarity') or {}).get('score') is not None else ""))


def build_globe(sites, start_lon=-60, start_lat=-10):
    """Rotatable orthographic globe (drag to spin) with one marker per site."""
    traces = []
    for body, color in (("Mars", MARS), ("Moon", CYAN)):
        group = [s for s in sites if s["analog_of"] == body]
        if not group:
            continue
        traces.append(go.Scattergeo(
            lat=[s["lat"] for s in group], lon=[s["lon"] for s in group],
            text=[s["name"].split(",")[0] for s in group],
            textposition=[LABEL_POS.get(s.get("id"), "top center") for s in group],
            textfont=dict(color=TEXT, size=11), mode="markers+text",
            marker=dict(size=11, color=color, line=dict(color=LIME, width=1.5), symbol="circle"),
            hovertext=[_hover(s) for s in group], hoverinfo="text",
            customdata=[s.get("viewer_3d") or "" for s in group],
            name=f"{body} analogs"))
    fig = go.Figure(traces)
    fig.update_geos(
        projection=dict(type="orthographic", rotation=dict(lon=start_lon, lat=start_lat)),
        showland=True, landcolor="#1B2B47", showocean=True, oceancolor="#07142B",
        showcoastlines=True, coastlinecolor="#3B5B7A", showcountries=True, countrycolor="#22324F",
        showlakes=True, lakecolor="#07142B", bgcolor=BG, resolution=50,
        lataxis=dict(showgrid=True, gridcolor="#12233F", dtick=15),
        lonaxis=dict(showgrid=True, gridcolor="#12233F", dtick=15))
    lons = [start_lon + d for d in range(0, 361, 6)]
    fig.frames = [go.Frame(layout=dict(geo=dict(projection=dict(rotation=dict(lon=lo, lat=start_lat)))),
                           name=str(i)) for i, lo in enumerate(lons)]
    fig.update_layout(
        title=dict(text="<b>NOVA TERRA LABS</b>  ·  Earth analog candidates  ·  drag to rotate, scroll to zoom",
                   font=dict(color=TEXT, size=16), x=0.01, y=0.97),
        paper_bgcolor=BG, font=dict(color=TEXT), height=720, margin=dict(l=0, r=0, t=60, b=10),
        legend=dict(x=0.01, y=0.92, bgcolor="rgba(11,20,40,.8)", bordercolor="#1C2A44", borderwidth=1),
        updatemenus=[dict(
            type="buttons", direction="right", x=0.99, y=0.97, xanchor="right", yanchor="top",
            bgcolor=PANEL, bordercolor=LIME, font=dict(color=BTN, size=12),
            buttons=[dict(label="▶ Rotate", method="animate",
                          args=[None, dict(frame=dict(duration=120, redraw=True), transition=dict(duration=0),
                                           fromcurrent=True, mode="immediate")]),
                     dict(label="❚❚ Stop", method="animate",
                          args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])])
    return fig


def write_globe_page(fig, path):
    """Offline page; clicking a marker opens that site's 3D terrain viewer."""
    click_js = """
var gd = document.getElementsByClassName('plotly-graph-div')[0];
gd.on('plotly_click', function(ev){
  var url = ev.points[0].customdata;
  if (url) { window.location.href = url.split('/').pop(); }
});"""
    # bundle the world map (sane-topojson, MIT) next to the page so it also works offline
    config = {"displaylogo": False, "responsive": True, "scrollZoom": True}
    topo_src = os.path.join(os.path.dirname(__file__), "assets", "topojson")
    if os.path.isdir(topo_src):        # otherwise Plotly loads the map from its CDN (needs internet)
        topo_dst = os.path.join(os.path.dirname(os.path.abspath(path)), "topojson")
        shutil.copytree(topo_src, topo_dst, dirs_exist_ok=True)
        config["topojsonURL"] = "topojson/"
    fig.write_html(path, include_plotlyjs="directory", full_html=True, post_script=click_js,
                   default_width="100%", default_height="100vh", config=config)
    with open(path, encoding="utf-8") as fh:
        page = fh.read()
    page = page.replace("<head>", "<head><title>NOVA TERRA LABS · Globe</title>"
                        f"<style>html,body{{margin:0;height:100%;background:{BG};overflow:hidden}}</style>", 1)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(page)


In [ ]:
%%writefile backend/app/targets.py
"""Planetary targets (Mars, Moon) measured with the same terrain method used on Earth.

Terrain: slope and roughness are computed from real orbital DEMs, resampled to ~30 m so that
they are comparable with the Copernicus DEM used for Earth sites.
Climate: values come from published measurements (cited in SOURCES) and are approximate.
"""
import math

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window

from .sites import HALF_KM
from .terrain import metrics_from_grid

TARGET_RES_M = 30.0   # common resolution for Earth / Mars / Moon terrain metrics
CACHE_DIR = None      # set by the pipeline: planetary DEMs are downloaded once and read locally


def _local_copy(url):
    """Downloads the whole file once (planetary DEMs are 10–40 MB) and returns a local path."""
    if not CACHE_DIR or not url.startswith("http"):
        return url
    import os
    import requests
    path = os.path.join(CACHE_DIR, url.rsplit("/", 1)[-1])
    if not os.path.exists(path):
        tmp = path + ".part"
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(tmp, "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        os.replace(tmp, path)
    return path

TARGETS = {
    "jezero": {
        "id": "jezero", "body": "Mars",
        "name": "Jezero crater — Perseverance landing site",
        "lat": 18.4447, "lon": 77.4508,
        "dem_url": "https://planetarymaps.usgs.gov/mosaic/mars2020_trn/CTX/"
                   "JEZ_ctx_B_soc_008_DTM_MOLAtopography_DeltaGeoid_20m_Eqc_latTs0_lon0.tif",
        "dem_source": "usgs-ctx-jezero-dtm",
        "climate": {"temp_range_c": 55.0, "precip_annual_mm": 0.0},
        "climate_source": "mars2020-meda",
        "notes": "Temperature range = mid-point of the 50–60 °C daily air-temperature variation reported "
                 "from Perseverance/MEDA. No liquid precipitation on Mars (0 mm).",
    },
    "lunar-south-pole": {
        "id": "lunar-south-pole", "body": "Moon",
        "name": "Lunar south pole — Shackleton rim (LOLA Site 04)",
        "lat": None, "lon": None,
        "dem_url": "https://pgda.gsfc.nasa.gov/data/LOLA_5mpp/Site04/Site04_final_adj_5mpp_surf.tif",
        "dem_source": "lola-south-pole-5mpp",
        "climate": {"temp_range_c": 242.0, "precip_annual_mm": 0.0},
        "climate_source": "nasa-lunar-environment",
        "diviner": True,
        "notes": "Temperature range = +22 °C to −220 °C reported by NASA for candidate south-pole landing "
                 "sites (surface temperature; the Moon has no air). No precipitation (0 mm).",
    },
}


def _read_resampled(src, window, res_m):
    """Reads a window and averages it down to ~res_m metres per pixel."""
    px = abs(src.transform.a)
    factor = max(1.0, res_m / px)
    out_h = max(10, int(round(window.height / factor)))
    out_w = max(10, int(round(window.width / factor)))
    z = src.read(1, window=window, out_shape=(out_h, out_w), resampling=Resampling.average,
                 masked=True).astype("float64").filled(np.nan)
    z[(z < -1e6) | (z > 1e6)] = np.nan
    dx = window.width * px / out_w
    dy = window.height * abs(src.transform.e) / out_h
    return z, dx, dy


def load_target_dem(target, half_km=HALF_KM, res_m=TARGET_RES_M):
    """Returns (z, dx, dy) for a 2*half_km square at the target, in metres."""
    with rasterio.open(_local_copy(target["dem_url"])) as src:
        half_px = int(round(half_km * 1000 / abs(src.transform.a)))
        if target["lat"] is not None:
            # projected planetary CRS (e.g. Mars equirectangular): convert lon/lat to x/y
            from pyproj import CRS, Transformer
            crs = CRS.from_user_input(src.crs)
            tr = Transformer.from_crs(crs.geodetic_crs, crs, always_xy=True)
            x, y = tr.transform(target["lon"], target["lat"])
            row, col = src.index(x, y)
            center = (target["lat"], target["lon"])
        else:
            # site-centred DEM: use the centre of the raster
            row, col = src.height // 2, src.width // 2
            center = None
            try:
                from pyproj import CRS, Transformer
                crs = CRS.from_user_input(src.crs)
                tr = Transformer.from_crs(crs, crs.geodetic_crs, always_xy=True)
                lon, lat = tr.transform(*src.xy(row, col))
                center = (lat, lon)
            except Exception:
                pass
        win = Window(col - half_px, row - half_px, 2 * half_px, 2 * half_px).intersection(
            Window(0, 0, src.width, src.height))
        z, dx, dy = _read_resampled(src, win, res_m)
        return z, dx, dy, center


def target_metrics(target, loader=load_target_dem):
    out = loader(target)
    z, dx, dy = out[:3]
    center = out[3] if len(out) > 3 else None
    metrics, profile = metrics_from_grid(z, dx, dy)
    metrics.update(target["climate"])
    info = {"dem_pixel_m": round((dx + dy) / 2, 1)}
    if center:
        info["center_lat"], info["center_lon"] = round(center[0], 4), round(center[1], 4)
    return metrics, profile, info


DIVINER_PRP_SOUTH = ("https://pds-geosciences.wustl.edu/lro/urn-nasa-pds-lro_diviner_derived1/"
                     "data_derived_prp/dlre_prp_south.tab")
MOON_R_KM = 1737.4


def diviner_prp_at(lat, lon, radius_km=2.0, url=DIVINER_PRP_SOUTH, lines=None):
    """Mean Diviner annual max surface temperature and annual average temperature (°C)
    for mesh cells within radius_km of (lat, lon). Streams the ~600 MB table once.

    Fixed-width columns from the PDS4 label: tri_clon 127–141, tri_clat 144–158,
    temp_avg 178–185 (K, 2 cm depth), temp_max 188–195 (K, surface).
    """
    import requests
    lat0, lon0 = math.radians(lat), math.radians(lon)
    tmax, tavg = [], []
    if lines is None:
        resp = requests.get(url, stream=True, timeout=600)
        resp.raise_for_status()
        lines = (ln.decode("ascii", "ignore") for ln in resp.iter_lines())
    for ln in lines:
        try:
            clat = float(ln[143:158])
            if abs(clat - lat) > 0.5:   # quick reject: far from the target latitude
                continue
            clon = float(ln[126:141])
            la, lo = math.radians(clat), math.radians(clon)
            d = MOON_R_KM * math.acos(max(-1.0, min(1.0, math.sin(lat0) * math.sin(la)
                                                   + math.cos(lat0) * math.cos(la) * math.cos(lo - lon0))))
            if d <= radius_km:
                tavg.append(float(ln[177:185]))
                tmax.append(float(ln[187:195]))
        except ValueError:
            continue
    if not tmax:
        raise RuntimeError("no Diviner cells near the target")
    k = 273.15
    return {"diviner_temp_max_c": sum(tmax) / len(tmax) - k,
            "diviner_temp_avg_c": sum(tavg) / len(tavg) - k,
            "n_cells": len(tmax)}


In [ ]:
%%writefile backend/app/similarity.py
"""Similarity index between an Earth site and its planetary target (0–100).

Each variable gives a partial score between 0 and 1; the index is their weighted average.

- slope_mean_deg, roughness_m, temp_range_c: ratio of the smaller to the larger value
  (1 = identical, 0.5 = one is twice the other). Scale-free, so Earth, Mars and Moon can be compared.
- precip_annual_mm (aridity): Mars and Moon have no rain, so the score is exp(-P / 200):
  10 mm/yr -> 0.95, 100 mm/yr -> 0.61, 500 mm/yr -> 0.08.

WEIGHTS are a methodological choice of the team and can be changed here.
"""
import math

# Team decision (14 Nov 2026): terrain and aridity dominate; temperature counts less because no Earth
# site reaches Martian or lunar thermal extremes.
WEIGHTS = {"slope_mean_deg": 0.30, "roughness_m": 0.30, "precip_annual_mm": 0.30, "temp_range_c": 0.10}

LABELS = {"slope_mean_deg": "slope", "roughness_m": "roughness",
          "precip_annual_mm": "aridity", "temp_range_c": "daily temperature range"}

ARIDITY_SCALE_MM = 200.0


def _ratio(a, b):
    a, b = abs(a), abs(b)
    if a == 0 and b == 0:
        return 1.0
    return min(a, b) / max(a, b)


def partial_scores(site_metrics, target_metrics):
    out = {}
    for var in WEIGHTS:
        if var not in site_metrics or var not in target_metrics:
            continue
        s, t = site_metrics[var], target_metrics[var]
        if var == "precip_annual_mm":
            out[var] = math.exp(-max(s, 0.0) / ARIDITY_SCALE_MM)
        else:
            out[var] = _ratio(s, t)
    return out


def similarity(site_metrics, target_metrics, target_name):
    parts = partial_scores(site_metrics, target_metrics)
    if not parts:
        return {"score": None, "components": {}, "explanation": "Not enough data to compare."}
    wsum = sum(WEIGHTS[v] for v in parts)
    score = 100 * sum(WEIGHTS[v] * parts[v] for v in parts) / wsum
    best = max(parts, key=parts.get)
    worst = min(parts, key=parts.get)
    missing = [LABELS[v] for v in WEIGHTS if v not in parts]
    expl = (f"Compared with {target_name}: most similar in {LABELS[best]} "
            f"({parts[best] * 100:.0f}/100), least similar in {LABELS[worst]} ({parts[worst] * 100:.0f}/100).")
    if missing:
        expl += " Not compared (missing data): " + ", ".join(missing) + "."
    return {"score": round(score, 2),
            "components": {v: round(parts[v] * 100, 2) for v in parts},
            "weights": {v: WEIGHTS[v] for v in parts},
            "explanation": expl}


In [ ]:
%%writefile backend/app/pipeline.py
"""NOVA TERRA LABS data pipeline.

Downloads real data for every candidate site, computes terrain and climate metrics,
drapes Sentinel-2 imagery over the DEM and writes:

    <out>/sites.json            data contract for the front-end
    <out>/viewer/index.html     table of sites with links to the 3D viewers
    <out>/viewer/<site>.html    interactive 3D terrain viewer per site (offline)
    <out>/viewer/globe.html     3D globe with all sites (click a marker to open its viewer)

Usage (from the backend/ folder):
    python -m app.pipeline                      # all sites
    python -m app.pipeline --sites andahua,mauna-kea
    python -m app.pipeline --no-imagery         # faster, elevation colouring only
"""
import argparse
import json
import os
import time
from datetime import datetime, timezone

import numpy as np
from affine import Affine

from .climate import climate_metrics
from .similarity import WEIGHTS, similarity
from .targets import TARGETS, diviner_prp_at, target_metrics
from .imagery import fetch_true_color
from .sites import CLIMATE_END, CLIMATE_START, HALF_KM, SITES
from .terrain import bbox, load_dem, terrain_metrics
from .viewer import build_figure, build_globe, write_globe_page, write_index, write_site_page

DECIMALS = 2

SOURCE_OF = {
    **{k: "copernicus-dem" for k in
       ("elevation_mean_m", "elevation_min_m", "elevation_max_m", "slope_mean_deg", "roughness_m")},
}
CLIMATE_KEYS = ("temp_max_mean_c", "temp_min_mean_c", "temp_range_c", "temp_abs_min_c",
                "temp_abs_max_c", "precip_annual_mm")

SOURCES = {
    "copernicus-dem": {
        "title": "Copernicus DEM GLO-30 (Copernicus Global Digital Elevation Model, 30 m)",
        "provider": "European Space Agency / Airbus",
        "doi": "https://doi.org/10.5270/ESA-c5d3d65",
        "access": "https://registry.opendata.aws/copernicus-dem/",
        "resolution": "30 m",
    },
    "nasa-power": {
        "title": "NASA POWER Daily API — T2M_MAX, T2M_MIN, PRECTOTCORR (2015–2024)",
        "provider": "NASA Langley Research Center POWER Project; meteorology from NASA GMAO MERRA-2",
        "access": "https://power.larc.nasa.gov/docs/services/api/temporal/daily/",
        "period": f"{CLIMATE_START} to {CLIMATE_END}",
        "resolution": "≈50 km (MERRA-2 grid); temperatures adjusted to site elevation with 6.5 °C/km",
    },
    "diviner-prp": {
        "title": "LRO Diviner Polar Resource Product, south (DLRE_PRP_SOUTH.TAB)",
        "provider": "NASA LRO Diviner — Paige et al. (2010), Science 330, 479–482",
        "doi": "https://doi.org/10.1126/science.1187726",
        "access": "https://pds-geosciences.wustl.edu/lro/urn-nasa-pds-lro_diviner_derived1/data_derived_prp/",
        "use": "annual maximum surface temperature and annual average temperature (2 cm depth) at the lunar target; context only, not used in the similarity index",
    },
    "open-meteo-era5": {
        "title": "Open-Meteo Historical Weather API (ERA5 / ERA5-Land reanalysis)",
        "provider": "Open-Meteo; ECMWF Copernicus Climate Change Service",
        "reference": "Hersbach et al. (2020), The ERA5 global reanalysis, QJRMS. https://doi.org/10.1002/qj.3803",
        "access": "https://open-meteo.com/en/docs/historical-weather-api",
        "period": f"{CLIMATE_START} to {CLIMATE_END}",
        "resolution": "≈9–25 km",
    },
    "usgs-ctx-jezero-dtm": {
        "title": "Mars 2020 Terrain Relative Navigation CTX DTM Mosaic (Jezero crater, 20 m)",
        "provider": "USGS Astrogeology Science Center — Fergason, Galuszka, Hare, Mayer & Redding (2020)",
        "doi": "https://doi.org/10.5066/P906QQT8",
        "access": "https://astrogeology.usgs.gov/search/map/mars_2020_terrain_relative_navigation_ctx_dtm_mosaic",
        "resolution": "20 m (resampled to 30 m for comparison)",
    },
    "lola-south-pole-5mpp": {
        "title": "High-Resolution LOLA Topography for Lunar South Pole Sites (Site 04, Shackleton rim)",
        "provider": "NASA GSFC Planetary Geodesy Data Archive — Barker et al. (2021), Planetary & Space Science 203, 105119",
        "doi": "https://doi.org/10.1016/j.pss.2020.105119",
        "access": "https://pgda.gsfc.nasa.gov/products/78",
        "resolution": "5 m (resampled to 30 m for comparison)",
    },
    "mars2020-meda": {
        "title": "Perseverance MEDA near-surface meteorology at Jezero crater",
        "provider": "Rodriguez-Manfredi et al. (2023), Nature Geoscience 16, 19–28",
        "doi": "https://doi.org/10.1038/s41561-022-01084-0",
        "use": "daily air-temperature variation of ~50–60 °C (mid-point 55 °C used)",
    },
    "nasa-lunar-environment": {
        "title": "Lunar Environment and Lunar Power Needs (NASA Technical Reports Server)",
        "provider": "NASA",
        "access": "https://ntrs.nasa.gov/citations/20205002224",
        "use": "candidate south-pole landing sites experience +22 °C to −220 °C (range 242 °C)",
    },
    "sentinel-2-l2a": {
        "title": "Sentinel-2 Level-2A surface reflectance, true-colour (TCI) image",
        "provider": "European Space Agency / Copernicus; COGs hosted on AWS (Element 84 Earth Search)",
        "access": "https://registry.opendata.aws/sentinel-2-l2a-cogs/",
        "resolution": "10 m",
        "use": "visual context for the 3D viewer only; not used in metrics",
    },
}


# ---- small cache so re-runs do not download everything again -------------------
def _cache_path(cache_dir, site, kind):
    """Cache file name includes the coordinates, so moving a site invalidates its cache."""
    return os.path.join(cache_dir, f"{site['id']}_{site['lat']:.4f}_{site['lon']:.4f}_{kind}.npz")


def cache_file(site, kind, cache_dir="data/cache"):
    """Public helper (used by the notebook) to find a site's cached DEM ('dem') or imagery ('rgb')."""
    return _cache_path(cache_dir, site, kind)


def _cached_dem(site, cache_dir):
    p = _cache_path(cache_dir, site, "dem")
    if os.path.exists(p):
        d = np.load(p)
        return d["z"], Affine(*d["transform"][:6])
    z, tr = load_dem(site["lat"], site["lon"])
    np.savez_compressed(p, z=z, transform=np.array(tr))
    return z, tr


def _cached_imagery(site, shape_hw, cache_dir):
    p = _cache_path(cache_dir, site, "rgb")
    if os.path.exists(p):
        d = np.load(p, allow_pickle=False)
        if tuple(d["rgb"].shape[1:]) == tuple(shape_hw):
            return d["rgb"], json.loads(str(d["meta"]))
    rgb, meta = fetch_true_color(bbox(site["lat"], site["lon"]), shape_hw)
    np.savez_compressed(p, rgb=rgb, meta=json.dumps(meta))
    return rgb, meta


def process_site(site, out_viewer, cache_dir, imagery=True, climate=True):
    print(f"▶ {site['name']}")
    raw, profile, errors, img_meta, viewer = {}, None, [], None, None
    z = tr = None
    try:
        z, tr = _cached_dem(site, cache_dir)
        metrics, profile = terrain_metrics(z, tr, site["lat"])
        raw.update(metrics)
    except Exception as ex:
        errors.append(f"DEM: {ex}")
    climate_src, climate_info = {}, {}
    if climate:
        try:
            cm, climate_src, climate_info = climate_metrics(site["lat"], site["lon"], raw.get("elevation_mean_m"))
            raw.update(cm)
            time.sleep(2)  # be gentle with the free APIs
        except Exception as ex:
            errors.append(f"Climate: {ex}")
    if z is not None:
        rgb = None
        if imagery:
            try:
                rgb, img_meta = _cached_imagery(site, z.shape, cache_dir)
            except Exception as ex:
                errors.append(f"Imagery: {ex}")
        try:
            fig = build_figure(site, z, tr, rgb, img_meta)
            fname = f"{site['id']}.html"
            write_site_page(fig, os.path.join(out_viewer, fname))
            viewer = f"viewer/{fname}"
        except Exception as ex:
            errors.append(f"3D viewer: {ex}")
    for e in errors:
        print("   ⚠", e)
    metrics = {k: {"value": round(v, DECIMALS), "source": climate_src.get(k, "nasa-power") if k in CLIMATE_KEYS else SOURCE_OF[k],
                   "illustrative": False}
               for k, v in raw.items()}
    sources = {m["source"] for m in metrics.values()}
    if img_meta:
        sources.add("sentinel-2-l2a")
    return {
        **{k: site[k] for k in ("id", "name", "country", "lat", "lon", "analog_of", "target")},
        "description": "",
        "metrics": metrics,
        "terrain_profile": profile,
        "imagery": img_meta,
        "climate_info": climate_info,
        "viewer_3d": viewer,
        "similarity": {"score": None, "explanation": "Pending: requires planetary target metrics"},
        "sources": sorted(sources),
        "errors": errors,
    }


def compute_targets(cache_dir):
    """Terrain + climate metrics for each planetary target (cached as JSON)."""
    from . import targets as targets_mod
    targets_mod.CACHE_DIR = cache_dir
    out = {}
    for tid, t in TARGETS.items():
        p = os.path.join(cache_dir, f"target_{tid}.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as fh:
                out[tid] = json.load(fh)
            continue
        print(f"▶ target: {t['name']}")
        rec = {k: t[k] for k in ("id", "body", "name", "lat", "lon", "notes")}
        rec["sources"] = sorted({t["dem_source"], t["climate_source"]})
        try:
            metrics, profile, info = target_metrics(t)
            src_of = {k: (t["climate_source"] if k in t["climate"] else t["dem_source"]) for k in metrics}
            rec["metrics"] = {k: {"value": round(v, DECIMALS), "source": src_of[k], "illustrative": False}
                              for k, v in metrics.items()}
            rec["terrain_profile"] = profile
            rec.update(info)
            if t.get("diviner"):
                try:
                    dv = diviner_prp_at(info["center_lat"], info["center_lon"])
                    rec["metrics"].update({k: {"value": round(v, DECIMALS), "source": "diviner-prp",
                                               "illustrative": False} for k, v in dv.items() if k != "n_cells"})
                    rec["diviner_cells"] = dv["n_cells"]
                    rec["sources"] = sorted(set(rec["sources"]) | {"diviner-prp"})
                except Exception as ex:
                    print(f"   ⚠ Diviner: {ex}")
                    rec.setdefault("warnings", []).append(f"Diviner: {ex}")
            rec["errors"] = []
            with open(p, "w", encoding="utf-8") as fh:
                json.dump(rec, fh)
        except Exception as ex:
            print(f"   ⚠ {ex}")
            rec["metrics"] = {k: {"value": v, "source": t["climate_source"], "illustrative": False}
                              for k, v in t["climate"].items()}
            rec["errors"] = [f"Target DEM: {ex}"]
        out[tid] = rec
    return out


def add_similarity(results, targets):
    for r in results:
        t = targets.get(r["target"])
        if not t:
            continue
        sm = {k: v["value"] for k, v in r["metrics"].items()}
        tm = {k: v["value"] for k, v in t["metrics"].items()}
        r["similarity"] = {"target": t["id"], **similarity(sm, tm, t["name"])}
    return results


def run(out_dir="data", site_ids=None, imagery=True, climate=True):
    out_viewer = os.path.join(out_dir, "viewer")
    cache_dir = os.path.join(out_dir, "cache")
    os.makedirs(out_viewer, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)
    sites = [s for s in SITES if not site_ids or s["id"] in site_ids]
    results = [process_site(s, out_viewer, cache_dir, imagery, climate) for s in sites]
    targets = compute_targets(cache_dir)
    add_similarity(results, targets)
    output = {
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "analysis_area_km": HALF_KM * 2,
        "sources": SOURCES,
        "known_limitations": [
            "The same method and sources are applied to all sites. Climate comes from a global gridded product "
            "(NASA POWER/MERRA-2, ≈50 km); very local extremes such as the hyperaridity of La Pampa de la Joya may be "
            "overestimated. No site-specific corrections are applied.",
            "The lunar target square (10 × 10 km around LOLA Site 04, Shackleton rim) includes part of the crater's inner "
            "wall (≈4 km of relief), so its mean slope is high; it represents the terrain surrounding a possible base, "
            "not only the flat rim.",
            "Temperatures are adjusted to the site elevation with a standard lapse rate (6.5 °C/km).",
            "Slope and roughness depend on DEM resolution; all bodies are compared at ≈30 m.",
            "Planetary temperature ranges are published reference values, not computed from rasters."],
        "similarity_method": {"weights": WEIGHTS,
                              "description": "Weighted average of partial scores (0-100). Slope, roughness and "
                                             "temperature range: ratio smaller/larger value. Aridity: exp(-P/200 mm)."},
        "targets": targets,
        "sites": results,
    }
    with open(os.path.join(out_dir, "sites.json"), "w", encoding="utf-8") as fh:
        json.dump(output, fh, ensure_ascii=False, indent=2)
    write_index(results, os.path.join(out_viewer, "index.html"))
    write_globe_page(build_globe(results), os.path.join(out_viewer, "globe.html"))
    ok = sum(not r["errors"] for r in results)
    print(f"\nsites.json + viewers written to {os.path.abspath(out_dir)}  ({ok}/{len(results)} sites without warnings)")
    print(f"Open {os.path.abspath(os.path.join(out_viewer, 'index.html'))} in your browser.")
    return results


def main():
    ap = argparse.ArgumentParser(description="NOVA TERRA LABS data pipeline")
    ap.add_argument("--out", default="data", help="output folder (default: data)")
    ap.add_argument("--sites", help="comma-separated site ids (default: all)")
    ap.add_argument("--no-imagery", action="store_true", help="skip Sentinel-2 imagery")
    ap.add_argument("--no-climate", action="store_true", help="skip Open-Meteo climate")
    a = ap.parse_args()
    run(a.out, a.sites.split(",") if a.sites else None, not a.no_imagery, not a.no_climate)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile backend/requirements.txt
numpy>=1.26
pandas>=2.1
requests>=2.31
rasterio>=1.3.9
scipy>=1.11
plotly>=5.20
pystac-client>=0.7
shapely>=2.0
pytest>=8.0
pyproj>=3.6


In [ ]:
%%writefile backend/README.md
# NOVA TERRA LABS — Backend data pipeline

Generates the scientific data used by the front-end (`sites.json`) and an interactive
3D terrain viewer for every candidate analog site. Everything runs locally with free,
open data — **no API keys or accounts are required**.

## Data sources

| Source | Used for | Access |
|---|---|---|
| Copernicus DEM GLO-30 (ESA, 30 m) — [doi:10.5270/ESA-c5d3d65](https://doi.org/10.5270/ESA-c5d3d65) | elevation, slope, roughness, terrain profile, 3D relief | public COGs on AWS |
| NASA POWER Daily API (MERRA-2, NASA GMAO) | temperature, daily range, annual precipitation (2015–2024) | free public API, no key (Open-Meteo/ERA5 used only as fallback) |
| USGS Mars 2020 CTX DTM (Jezero, 20 m) and NASA LOLA 5 m DEM (Shackleton rim) | slope and roughness of the planetary targets, same method as on Earth | public downloads |
| LRO Diviner Polar Resource Product (NASA PDS) | annual max / average temperature at the lunar target (context) | public download (~600 MB, cached) |
| Sentinel-2 L2A true colour (Copernicus/ESA) | satellite image draped on the 3D relief (visual only) | public COGs on AWS via Earth Search STAC |

## Setup

Requires Python 3.10+.

```bash
cd backend
python -m venv .venv
# Windows: .venv\Scripts\activate      macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
```

## Run

```bash
python -m app.pipeline                              # all sites
python -m app.pipeline --sites andahua,mauna-kea    # only some sites
python -m app.pipeline --no-imagery                 # faster: elevation colouring only
```

Outputs (in `backend/data/`):

| File | Content |
|---|---|
| `sites.json` | data contract for the front-end: metrics (2 decimals), source of each value, terrain profile, imagery scene, link to the 3D viewer |
| `viewer/index.html` | table of all sites with links to their 3D viewers — open it in a browser |
| `viewer/globe.html` | rotatable 3D globe with all sites; click a marker to open its terrain viewer |
| `viewer/<site-id>.html` | interactive 3D viewer (rotate, zoom, satellite/elevation, vertical exaggeration ×1–×10); works offline |
| `cache/` | downloaded DEM and imagery, so re-runs are fast (not committed to Git) |

## Third-party assets

- `app/assets/topojson/`: world map for the globe, from [sane-topojson](https://github.com/etpinard/sane-topojson) (MIT licence), bundled so the globe works offline.

## Tests

```bash
python -m pytest -q
```

## Sites

Defined in `app/sites.py`. Coordinates are approximate centre points of a 10 × 10 km
analysis square and must be validated before the final submission.

## Known limitations

- The lunar target square (10 × 10 km around the Shackleton rim, LOLA Site 04) includes part of the crater's inner wall, so its mean slope is high: it represents the terrain around a possible base, not only the flat rim.

- The same method and sources are applied to all sites. Climate comes from global gridded products, so very local extremes (e.g. the hyperaridity of La Pampa de la Joya) may be overestimated; no site-specific corrections are applied.

- Climate values come from a reanalysis grid (≈9–25 km): they describe the region, not the exact point.
- Slope and roughness depend on DEM resolution; comparisons with Moon/Mars data must use equivalent resolutions.
- Sentinel-2 coverage is limited at very high latitudes; when no clear scene is found the viewer falls back to elevation colouring and says so.
- Vertical exaggeration in the viewer is visual only; hover always shows true elevation.


In [ ]:
%%writefile backend/tests/__init__.py



In [ ]:
%%writefile backend/tests/test_terrain.py
import math

import numpy as np
from affine import Affine

from app.terrain import bbox, terrain_metrics, tile_name
from app.viewer import build_figure

LAT = -16.6
RES = 1 / 3600
TR = Affine(RES, 0, -72, 0, -RES, -16.5)


def _plane(slope_ratio=0.1, ny=300, nx=320):
    dx = RES * 111320 * math.cos(math.radians(LAT))
    return 2000 + slope_ratio * dx * np.arange(nx)[None, :].repeat(ny, 0)


def test_tile_names():
    assert tile_name(-17, -72) == "Copernicus_DSM_COG_10_S17_00_W072_00_DEM"
    assert tile_name(75, -90) == "Copernicus_DSM_COG_10_N75_00_W090_00_DEM"


def test_bbox_is_10km():
    w, s, e, n = bbox(LAT, -71.95)
    assert abs((n - s) * 110.574 - 10) < 1e-6


def test_slope_of_10_percent_plane():
    m, profile = terrain_metrics(_plane(), TR, LAT)
    assert abs(m["slope_mean_deg"] - math.degrees(math.atan(0.1))) < 0.01
    assert m["roughness_m"] < 1e-6
    assert len(profile["elevation_m"]) == 100


def test_roughness_detects_noise():
    z = _plane() + np.random.default_rng(0).normal(0, 2, (300, 320))
    m, _ = terrain_metrics(z, TR, LAT)
    assert 1.5 < m["roughness_m"] < 2.5


def test_viewer_with_and_without_imagery():
    site = {"name": "Test", "country": "Peru", "analog_of": "Mars", "lat": LAT, "lon": -72}
    z = _plane()
    rgb = np.full((3,) + z.shape, 120, dtype="uint8")
    fig = build_figure(site, z, TR, rgb, {"date": "2025-01-01", "scene_id": "X"})
    assert len(fig.data) == 2 and fig.data[0].visible is not False
    fig2 = build_figure(site, z, TR, None, None)
    assert len(fig2.data) == 1


def test_globe_has_mars_and_moon_groups():
    from app.viewer import build_globe
    sites = [{"name": "A", "country": "Peru", "analog_of": "Mars", "lat": -16, "lon": -72, "metrics": {}},
             {"name": "B", "country": "USA", "analog_of": "Moon", "lat": 35, "lon": -111, "metrics": {}}]
    fig = build_globe(sites)
    assert [t.name for t in fig.data] == ["Mars analogs", "Moon analogs"]
    assert len(fig.frames) > 10


def test_similarity_identical_and_wet():
    from app.similarity import similarity
    t = {"slope_mean_deg": 5, "roughness_m": 2, "precip_annual_mm": 0, "temp_range_c": 55}
    same = similarity(dict(t), t, "T")
    assert same["score"] == 100
    wet = similarity({**t, "precip_annual_mm": 2000}, t, "T")
    assert wet["score"] < 80 and "aridity" in wet["explanation"]


def test_target_metrics_use_same_method():
    from app.targets import TARGETS, target_metrics
    z = _plane()
    m, profile, info = target_metrics(TARGETS["jezero"], loader=lambda t: (z, 30.0, 30.0))
    assert abs(m["slope_mean_deg"] - math.degrees(math.atan(0.1 * 30 / (RES * 111320 * math.cos(math.radians(LAT)))))) < 0.5
    assert m["precip_annual_mm"] == 0 and m["temp_range_c"] == 55


def test_diviner_fixed_width_parser():
    from app.targets import diviner_prp_at
    def row(clon, clat, tavg, tmax):
        line = [" "] * 210
        def put(start, length, val):
            txt = f"{val:>{length}}"
            line[start - 1:start - 1 + length] = list(txt)
        put(127, 15, f"{clon:.6f}"); put(144, 15, f"{clat:.6f}")
        put(178, 8, f"{tavg:.2f}"); put(188, 8, f"{tmax:.2f}")
        return "".join(line)
    lines = [row(200.0, -89.70, 100.0, 300.0), row(200.0, -89.71, 110.0, 310.0), row(20.0, -85.0, 50.0, 80.0)]
    out = diviner_prp_at(-89.70, 200.0, radius_km=2.0, lines=lines)
    assert out["n_cells"] == 2
    assert abs(out["diviner_temp_max_c"] - (305.0 - 273.15)) < 1e-6


def test_power_summary():
    from app.climate import _summarise
    m = _summarise([20, 22, None], [5, 6, 7], [0.0, 1.0, 0.0], ["2015-01-01", "2015-01-02", "2016-01-01"])
    assert abs(m["temp_range_c"] - 15.5) < 1e-9 and m["precip_annual_mm"] == 0.5


def test_lapse_rate_adjustment():
    from app import climate
    days = [f"2015010{i}" for i in range(1, 8)]
    def fake_power(lat, lon, parameters, community, utc=False):
        if "IMERG" in parameters:
            return {"IMERG_PRECTOT": {d: 0.0 for d in days}}, 600.0
        return {"T2M_MAX": {d: 20.0 for d in days}, "T2M_MIN": {d: 10.0 for d in days},
                "PRECTOTCORR": {d: 1.0 for d in days}}, 600.0
    climate._power = fake_power
    m, src, info = climate.nasa_power_metrics(19.8, -155.5, site_elev_m=3600.0)
    assert abs(info["temperature_lapse_adjustment_c"] + 19.5) < 1e-9
    assert abs(m["temp_max_mean_c"] - 0.5) < 1e-9 and abs(m["temp_range_c"] - 10.0) < 1e-9
    assert src["precip_annual_mm"] == "nasa-power" and abs(m["precip_annual_mm"] - 7.0) < 1e-9


## Run the tests and the pipeline
Sites are defined in `backend/app/sites.py` (cell above). Coordinates are approximate centre points **to be validated by the team**.

In [ ]:
%cd backend
!python -m pytest -q

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
from app import pipeline
importlib.reload(pipeline)
results = pipeline.run("data")        # all 13 sites; already-downloaded data is reused from data/cache
# First run: the Diviner table (~600 MB) is streamed once for the lunar target (a few minutes); later runs use the cache.

## Verification
- **Map:** every 10 × 10 km square should sit on the intended site (zoom in).
- **Checks:** every site should show ✅. Imagery warnings at very high latitudes are expected (the viewer then uses elevation colouring).
- **Cross-check:** compare the mean elevation of 2–3 sites with Google Earth.

In [ ]:
import folium
from app.terrain import bbox
m = folium.Map(location=[20, -40], zoom_start=2,
               tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
               attr="Esri World Imagery")
for r in results:
    w, s, e, n = bbox(r["lat"], r["lon"])
    folium.Rectangle([[s, w], [n, e]], color="lime", weight=2, tooltip=r["name"]).add_to(m)
    folium.Marker([r["lat"], r["lon"]], tooltip=f'{r["name"]} ({r["analog_of"]})').add_to(m)
display(m)

for r in results:
    v = {k: x["value"] for k, x in r["metrics"].items()}
    warnings = list(r["errors"])
    if len(v) < 11: warnings.append(f"missing metrics ({len(v)}/11)")
    if v.get("elevation_min_m") is not None and not (v["elevation_min_m"] <= v["elevation_mean_m"] <= v["elevation_max_m"]):
        warnings.append("inconsistent elevations")
    if v.get("slope_mean_deg", 0) > 45: warnings.append("mean slope > 45°: review")
    if v.get("precip_annual_mm", 0) > 1500: warnings.append("very wet for an analog: review point")
    print(("✅ " if not warnings else "⚠️ ") + r["name"] + ("" if not warnings else " → " + " | ".join(warnings)))

## 3D globe
Drag to rotate, scroll to zoom, **▶ Rotate** to spin it. Hover a marker to see the site metrics. Orange = Mars analogs, cyan = Moon analogs.
(In the downloaded `viewer/globe.html`, clicking a marker opens that site's 3D terrain viewer.)

In [ ]:
from app.viewer import build_globe
build_globe(results).show()

## Results table
All metrics with 2 decimals. Orange/cyan colouring is only in the globe; here sites are grouped by analog type.

In [ ]:
import pandas as pd
LABELS = {"elevation_mean_m": "Mean elev. (m)", "elevation_min_m": "Min elev. (m)", "elevation_max_m": "Max elev. (m)",
          "slope_mean_deg": "Mean slope (°)", "roughness_m": "Roughness (m)", "temp_max_mean_c": "Mean max T (°C)",
          "temp_min_mean_c": "Mean min T (°C)", "temp_range_c": "Daily T range (°C)", "temp_abs_min_c": "Abs. min T (°C)",
          "temp_abs_max_c": "Abs. max T (°C)", "precip_annual_mm": "Precip. (mm/yr)",
          "diviner_temp_max_c": "Diviner annual max T (°C)", "diviner_temp_avg_c": "Diviner annual avg T (°C)"}
table = pd.DataFrame([{"Site": r["name"], "Country": r["country"], "Analog of": r["analog_of"],
                       **{LABELS[k]: v["value"] for k, v in r["metrics"].items()}} for r in results])
table = table.sort_values(["Analog of", "Site"]).set_index("Site")
table.style.format(precision=2, thousands=",", na_rep="—")

## Data sources (traceability)

In [ ]:
src = pd.DataFrame([{"Id": k, "Dataset": v["title"], "Provider": v["provider"],
                     "Reference / access": v.get("doi") or v.get("reference") or v.get("access"),
                     "Resolution": v.get("resolution", "")} for k, v in pipeline.SOURCES.items()]).set_index("Id")
img = pd.DataFrame([{"Site": r["name"],
                     "Sentinel-2 scene": (r["imagery"] or {}).get("scene_id", "not available"),
                     "Image date": (r["imagery"] or {}).get("date", "—"),
                     "Cloud cover (%)": (r["imagery"] or {}).get("cloud_cover_pct"),
                     "Warnings": "; ".join(r["errors"]) or "—"} for r in results]).set_index("Site")
display(src)
display(img.style.format(precision=2, na_rep="—"))

## Planetary targets and similarity index
Each Earth site is compared with its target using four variables. Terrain (slope, roughness) is computed **with the same method** on real orbital DEMs of Mars (USGS CTX, 20 m) and the Moon (NASA LOLA, 5 m), all resampled to 30 m. Climate targets come from published measurements (see sources).

**Partial scores (0–100):** slope, roughness and daily temperature range = smaller value ÷ larger value; aridity = exp(−precipitation / 200 mm).
**Index** = weighted average. The weights are a team decision and can be changed in `backend/app/similarity.py` (`WEIGHTS`).

In [ ]:
import json, plotly.graph_objects as go
from app.viewer import BG, TEXT
from app.similarity import WEIGHTS
print("Weights:", WEIGHTS)
with open("data/sites.json", encoding="utf-8") as fh:
    targets = json.load(fh)["targets"]
tt = pd.DataFrame([{"Target": t["name"], "Body": t["body"],
                    **{LABELS.get(k, k): v["value"] for k, v in t["metrics"].items()},
                    "Warnings": "; ".join(t.get("errors", [])) or "—"} for t in targets.values()]).set_index("Target")
display(tt.style.format(precision=2, thousands=",", na_rep="—"))

In [ ]:
rank = pd.DataFrame([{"Site": r["name"], "Analog of": r["analog_of"], "Target": targets[r["target"]]["name"],
                      "Similarity": r["similarity"].get("score"),
                      **{f"{k.split('_')[0].capitalize()} score": v for k, v in r["similarity"].get("components", {}).items()},
                      "Explanation": r["similarity"].get("explanation", "")} for r in results])
rank = rank.sort_values(["Analog of", "Similarity"], ascending=[True, False]).set_index("Site")
display(rank.style.format(precision=2, na_rep="—").background_gradient(subset=["Similarity"], cmap="YlGn"))

fig = go.Figure()
for body, color in (("Mars", "#FF7A45"), ("Moon", "#00D9FF")):
    g = rank[rank["Analog of"] == body].sort_values("Similarity")
    fig.add_trace(go.Bar(x=g["Similarity"], y=g.index, orientation="h", name=f"{body} analogs",
                         marker_color=color, text=[f"{v:.2f}" for v in g["Similarity"]], textposition="outside",
                         hovertext=g["Explanation"], hoverinfo="text"))
fig.update_layout(title="Similarity index (0–100) by site", height=520, paper_bgcolor=BG, plot_bgcolor=BG,
                  font=dict(color=TEXT), barmode="group", margin=dict(l=260, r=40, t=60, b=40),
                  xaxis=dict(range=[0, 105], gridcolor="#1C2A44", title="Similarity"), yaxis=dict(title=""))
fig.show()

## Site explorer
Pick a site in the menu: you get its metrics and similarity, a **satellite close-up** of the 10 × 10 km square with a link to open the site in **Google Earth (3D)**, the **3D terrain** (satellite / elevation, vertical exaggeration ×1–×10) and the **W→E terrain profile** through the centre of the square.

In [ ]:
import json, numpy as np, ipywidgets as widgets
import plotly.graph_objects as go
from affine import Affine
from IPython.display import display, HTML
from app.viewer import build_figure, BG, TEXT, LIME
from app.sites import SITES

def show_3d(site_id):
    site = next(s for s in SITES if s["id"] == site_id)
    d = np.load(pipeline.cache_file(site, "dem"))
    z, tr = d["z"], Affine(*d["transform"][:6])
    rgb = meta = None
    try:
        im = np.load(pipeline.cache_file(site, "rgb"))
        rgb, meta = im["rgb"], json.loads(str(im["meta"]))
    except FileNotFoundError:
        pass
    fig = build_figure(site, z, tr, rgb, meta)
    fig.update_layout(height=700)
    fig.show()

def show_profile(r):
    p = r["terrain_profile"]
    if not p:
        print("No terrain profile for this site."); return
    fig = go.Figure(go.Scatter(x=p["distance_km"], y=p["elevation_m"], mode="lines",
                               line=dict(color=LIME, width=2), fill="tozeroy", fillcolor="rgba(182,255,59,0.08)",
                               hovertemplate="%{x:.2f} km<br>%{y:.2f} m<extra></extra>"))
    lo, hi = min(p["elevation_m"]), max(p["elevation_m"])
    pad = max(10, (hi - lo) * 0.15)
    fig.update_layout(title=f"Terrain profile W → E · {r['name']}", height=320, paper_bgcolor=BG, plot_bgcolor=BG,
                      font=dict(color=TEXT), margin=dict(l=60, r=20, t=50, b=40),
                      xaxis=dict(title="Distance (km)", gridcolor="#1C2A44"),
                      yaxis=dict(title="Elevation (m)", gridcolor="#1C2A44", range=[lo - pad, hi + pad]))
    fig.show()

import folium
from app.terrain import bbox

def earth_link(lat, lon):
    # Google Earth web, centred on the site, ~15 km camera distance, tilted 65° for a 3D view
    return f"https://earth.google.com/web/@{lat},{lon},0a,15000d,35y,0h,65t,0r"

def site_closeup(r, zoom=13):
    m = folium.Map(location=[r["lat"], r["lon"]], zoom_start=zoom, height=420,
                   tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                   attr="Esri World Imagery")
    w, s_, e, n = bbox(r["lat"], r["lon"])
    folium.Rectangle([[s_, w], [n, e]], color="lime", weight=2, fill=False, tooltip="10 × 10 km analysis square").add_to(m)
    folium.Marker([r["lat"], r["lon"]], tooltip=r["name"]).add_to(m)
    display(HTML(f'<a href="{earth_link(r["lat"], r["lon"])}" target="_blank" '
                 f'style="color:#B7FF00;font-weight:600">Open {r["name"]} in Google Earth 3D ↗</a>'))
    display(m)

def explore(site_id):
    r = next(x for x in results if x["id"] == site_id)
    display(HTML(f"<h3 style='margin:4px 0'>{r['name']} · {r['country']} · analog of {r['analog_of']}</h3>"
                 f"<div>{r['lat']:.3f}, {r['lon']:.3f} · 10 × 10 km square</div>"))
    m = pd.DataFrame([{"Metric": LABELS[k], "Value": v["value"], "Source": v["source"]}
                      for k, v in r["metrics"].items()]).set_index("Metric")
    display(m.style.format({"Value": "{:,.2f}"}))
    sim = r.get("similarity") or {}
    if sim.get("score") is not None:
        display(HTML(f"<p><b>Similarity with {targets[r['target']]['name']}: {sim['score']:.2f} / 100</b><br>{sim['explanation']}</p>"))
    if r["errors"]:
        print("⚠ " + " | ".join(r["errors"]))
    site_closeup(r)
    show_3d(site_id)
    show_profile(r)

picker = widgets.Dropdown(options=[(f"{r['name']} ({r['analog_of']})", r["id"]) for r in results],
                          description="Site:", layout=widgets.Layout(width="520px"))
out = widgets.Output()
def on_change(change=None):
    with out:
        out.clear_output(wait=True)
        explore(picker.value)
picker.observe(on_change, names="value")
display(picker, out)
on_change()

## Preview of `sites.json` (first site)
This is exactly what the front-end receives. The terrain profile is shortened here for readability.

In [ ]:
with open("data/sites.json", encoding="utf-8") as fh:
    data = json.load(fh)
first = dict(data["sites"][0])
if first.get("terrain_profile"):
    first["terrain_profile"] = {k: v[:5] + ["..."] for k, v in first["terrain_profile"].items()}
print(json.dumps(first, ensure_ascii=False, indent=2))

## Save results to Google Drive
Saves `sites.json` and a zip with the viewers in **NovaTerra Labs / Resultados** on Drive, so the team (and Claude) can read them without downloading anything.
The first time, Colab asks you to authorize Google: choose **rlostaunau@gmail.com**. Each run replaces the previous files; Drive keeps the earlier versions in the file history.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

RESULTS_FOLDER_ID = "1V_eAK5An1qWfbhn92N4CXwzQ96Igccii"   # NovaTerra Labs / Resultados
auth.authenticate_user()
drive = build("drive", "v3")

def upsert(path, name, mime):
    q = f"name = '{name}' and '{RESULTS_FOLDER_ID}' in parents and trashed = false"
    found = drive.files().list(q=q, fields="files(id)", supportsAllDrives=True,
                               includeItemsFromAllDrives=True).execute().get("files", [])
    media = MediaFileUpload(path, mimetype=mime, resumable=True)
    if found:
        drive.files().update(fileId=found[0]["id"], media_body=media, supportsAllDrives=True).execute()
        return "updated"
    drive.files().create(body={"name": name, "parents": [RESULTS_FOLDER_ID]}, media_body=media,
                         fields="id", supportsAllDrives=True).execute()
    return "created"

!cd data && rm -f ../nova_terra_data.zip && zip -q -r ../nova_terra_data.zip sites.json viewer
print("sites.json:", upsert("data/sites.json", "sites.json", "application/json"))
print("nova_terra_data.zip:", upsert("nova_terra_data.zip", "nova_terra_data.zip", "application/zip"))
print("Done → https://drive.google.com/drive/folders/" + RESULTS_FOLDER_ID)

# each 3D viewer as its own file in Resultados/viewer (the zip is too large for some tools)
import glob, os
q = f"'{RESULTS_FOLDER_ID}' in parents and name='viewer' and mimeType='application/vnd.google-apps.folder' and trashed=false"
found = drive.files().list(q=q, fields="files(id)").execute()["files"]
viewer_id = found[0]["id"] if found else drive.files().create(
    body={"name": "viewer", "mimeType": "application/vnd.google-apps.folder", "parents": [RESULTS_FOLDER_ID]},
    fields="id").execute()["id"]
for f in sorted(glob.glob("data/viewer/*.html")):
    name = os.path.basename(f)
    if name in ("index.html", "globe.html"):
        continue
    media = MediaFileUpload(f, mimetype="text/html", resumable=True)
    old = drive.files().list(q=f"'{viewer_id}' in parents and name='{name}' and trashed=false", fields="files(id)").execute()["files"]
    if old:
        drive.files().update(fileId=old[0]["id"], media_body=media).execute()
    else:
        drive.files().create(body={"name": name, "parents": [viewer_id]}, media_body=media).execute()
    print("✓ viewer/" + name)

In [ ]:
# v17: export the Jezero and lunar-south-pole terrain (same 10 x 10 km square) for the side-by-side 3D view
import json, base64, numpy as np; from app import targets as T; T.CACHE_DIR = "data/cache"
def enc(k):
    z, dx, dy = T.load_target_dem(T.TARGETS[k])[:3]
    z = np.where(np.isfinite(z), z, np.nanmean(z))[::-1]          # rows south -> north
    s = max(1, int(np.ceil(max(z.shape) / 170))); z = z[::s, ::s]; zmin = float(z.min())
    return {"id": k, "nx": z.shape[1], "ny": z.shape[0], "x": [0, round(z.shape[1] * dx * s / 1000, 3)],
            "y": [0, round(z.shape[0] * dy * s / 1000, 3)], "zmin": zmin,
            "z_dm_u16": base64.b64encode(np.clip(np.round((z - zmin) * 10), 0, 65535).astype(np.uint16).tobytes()).decode()}
out = {k: enc(k) for k in T.TARGETS}
json.dump(out, open("data/targets_terrain.json", "w"), separators=(",", ":"))
print({k: (v["nx"], v["ny"], round(v["zmin"]), v["x"], v["y"]) for k, v in out.items()})
print("targets_terrain.json:", upsert("data/targets_terrain.json", "targets_terrain.json", "application/json"))

## Discover: search whole zones
The same index applied to every 10 × 10 km land cell of three search zones (southern Peru, Atacama–Altiplano in Chile, Puna de Catamarca in Argentina). Code: `backend/app/discover.py`. Output: `data/discover.json`.

In [ ]:
%%writefile app/discover.py
"""Discover: apply the same similarity index to every 10 x 10 km cell of a search zone.

The sites in sites.py were chosen by the team. This module searches a whole zone instead, so the
index can find places nobody proposed. Same data and method as the 15 sites:
- terrain: Copernicus DEM GLO-30 (slope, roughness), read once for the whole zone;
- climate: NASA POWER daily 2015-2024 (MERRA-2), fetched once per MERRA-2 grid cell
  (0.5 deg lat x 0.625 deg lon) and adjusted to each cell's mean elevation with 6.5 C/km;
- similarity: app.similarity against the Jezero and lunar-south-pole targets in data/sites.json.

Usage (from backend/, after the main pipeline has written data/sites.json):
    python -m app.discover --zone southern-peru
Output: data/discover.json (one entry per zone; other zones already in the file are kept).
"""
import argparse
import json
import math
import os
import time
from datetime import datetime, timezone

import numpy as np
import rasterio
from rasterio.merge import merge

from .climate import LAPSE_RATE_C_PER_KM, _power, _summarise
from .similarity import similarity
from .terrain import COP_URL, metrics_from_grid, tile_name

CELL_KM = 10.0
ZONES = {
    "southern-peru": {"name": "Southern Peru (Arequipa, Moquegua, Tacna)", "country": "Peru",
                      "south": -18.0, "north": -16.0, "west": -72.0, "east": -70.0},
    "atacama-chile": {"name": "Atacama and Altiplano, Chile", "country": "Chile",
                      "south": -24.0, "north": -22.0, "west": -69.0, "east": -67.0},
    "catamarca-argentina": {"name": "Puna de Catamarca, Argentina", "country": "Argentina",
                            "south": -27.0, "north": -25.0, "west": -68.0, "east": -66.0},
}
METRIC_KEYS = ("slope_mean_deg", "roughness_m", "temp_range_c", "precip_annual_mm")


def load_zone_dem(z):
    """Copernicus DEM for the whole zone (float32). Missing tiles (ocean) stay NaN."""
    srcs = []
    for la in range(math.floor(z["south"]), math.ceil(z["north"])):
        for lo in range(math.floor(z["west"]), math.ceil(z["east"])):
            try:
                srcs.append(rasterio.open(COP_URL.format(n=tile_name(la, lo))))
            except Exception:
                print(f"   no DEM tile {tile_name(la, lo)} (ocean)")
    try:
        arr, tr = merge(srcs, bounds=(z["west"], z["south"], z["east"], z["north"]), nodata=np.nan)
    finally:
        for s in srcs:
            s.close()
    dem = arr[0].astype("float32")
    dem[dem < -1000] = np.nan
    return dem, tr


def power_cell(lat, lon, cache_dir):
    """Raw NASA POWER series for the MERRA-2 cell containing (lat, lon), cached on disk."""
    glat, glon = round(lat * 2) / 2, round(lon / 0.625) * 0.625
    path = os.path.join(cache_dir, f"power_{glat:.3f}_{glon:.3f}.json")
    if os.path.exists(path):
        return json.load(open(path))
    t, grid_elev = _power(glat, glon, "T2M_MAX,T2M_MIN,PRECTOTCORR", "AG")
    out = {"t": t, "grid_elev": grid_elev}
    json.dump(out, open(path, "w"))
    time.sleep(1)
    return out


def climate_for(p, elev):
    t, ge = p["t"], p["grid_elev"]
    dates = sorted(t["T2M_MAX"])
    adj = -LAPSE_RATE_C_PER_KM * (elev - ge) / 1000.0 if ge is not None else 0.0
    tmax = [None if t["T2M_MAX"][d] is None else t["T2M_MAX"][d] + adj for d in dates]
    tmin = [None if t["T2M_MIN"][d] is None else t["T2M_MIN"][d] + adj for d in dates]
    return _summarise(tmax, tmin, [t["PRECTOTCORR"][d] for d in dates], dates)


def run(zone_id, data_dir="data", cache_dir="data/cache"):
    z = ZONES[zone_id]
    os.makedirs(cache_dir, exist_ok=True)
    sites = json.load(open(os.path.join(data_dir, "sites.json")))
    targets = {k: {m: v["value"] for m, v in t["metrics"].items()} for k, t in sites["targets"].items()}
    names = {k: t["name"] for k, t in sites["targets"].items()}
    print(f"Zone {zone_id}: reading Copernicus DEM ...")
    dem, tr = load_zone_dem(z)
    px_deg = abs(tr.a)
    dlat = CELL_KM / 110.574
    cells, skipped = [], 0
    lat = z["north"] - dlat / 2
    while lat - dlat / 2 >= z["south"] - 1e-9:
        dlon = CELL_KM / (111.320 * math.cos(math.radians(lat)))
        lon = z["west"] + dlon / 2
        while lon + dlon / 2 <= z["east"] + 1e-9:
            r0 = int((tr.f - (lat + dlat / 2)) / px_deg); r1 = int((tr.f - (lat - dlat / 2)) / px_deg)
            c0 = int(((lon - dlon / 2) - tr.c) / px_deg); c1 = int(((lon + dlon / 2) - tr.c) / px_deg)
            g = dem[max(r0, 0):r1, max(c0, 0):c1].astype("float64")
            land = np.isfinite(g) & (g > 0.5)          # Copernicus ocean = 0 m or missing tile
            if g.size == 0 or land.mean() < 0.5:
                skipped += 1
            else:
                g[~land] = np.nan
                dx = px_deg * 111320.0 * math.cos(math.radians(lat)); dy = px_deg * 110574.0
                tm, _ = metrics_from_grid(g, dx, dy)
                cm = climate_for(power_cell(lat, lon, cache_dir), tm["elevation_mean_m"])
                m = {**tm, **cm}
                sims = {k: similarity(m, targets[k], names[k]) for k in targets}
                cells.append({"lat": round(lat, 4), "lon": round(lon, 4),
                              "elev": round(tm["elevation_mean_m"]), "relief": round(tm["elevation_max_m"] - tm["elevation_min_m"]),
                              **{k: round(m[k], 2) for k in METRIC_KEYS},
                              "mars": sims["jezero"]["score"], "moon": sims["lunar-south-pole"]["score"],
                              "mars_c": sims["jezero"]["components"], "moon_c": sims["lunar-south-pole"]["components"]})
            lon += dlon
        lat -= dlat
        print(f"   row {lat:.2f}: {len(cells)} land cells so far")
    out_path = os.path.join(data_dir, "discover.json")
    out = json.load(open(out_path)) if os.path.exists(out_path) else {"zones": {}}
    out["cell_km"] = CELL_KM
    out["method"] = ("Same similarity index as the analogue sites, applied to every 10 x 10 km land cell of the zone. "
                     "Terrain: Copernicus DEM GLO-30. Climate: NASA POWER daily 2015-2024 (one MERRA-2 grid cell "
                     "shared by several 10 km cells), adjusted to each cell's mean elevation with 6.5 C/km.")
    out["zones"][zone_id] = {**z, "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                             "skipped_ocean_cells": skipped, "cells": cells}
    json.dump(out, open(out_path, "w"), separators=(",", ":"))
    best = lambda k: max(cells, key=lambda c: c[k])
    print(f"Done: {len(cells)} cells ({skipped} ocean). Best Mars {best('mars')['mars']} at "
          f"{best('mars')['lat']}, {best('mars')['lon']}; best Moon {best('moon')['moon']} at "
          f"{best('moon')['lat']}, {best('moon')['lon']} -> {out_path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--zone", default="southern-peru", choices=sorted(ZONES))
    run(ap.parse_args().zone)


In [ ]:
!python -m app.discover --zone southern-peru
!python -m app.discover --zone atacama-chile
!python -m app.discover --zone catamarca-argentina
print("discover.json:", upsert("data/discover.json", "discover.json", "application/json"))

## Optional: download everything
Only needed to open the pages on your computer or to hand `sites.json` to the front-end.

In [ ]:
# Download sites.json + all 3D viewers as a zip (open viewer/index.html on your computer)
!cd data && zip -q -r ../nova_terra_data.zip sites.json viewer
from google.colab import files
files.download("nova_terra_data.zip")